# 导入需要的库

In [1]:
import pandas as pd
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler

# 这一行负责从 preprocess.py 文件中加载 gen_model_input 函数
from preprocess import gen_data_set, gen_model_input 

# 处理数据

In [ ]:
# import random
# import csv
# import os
# import pandas as pd
# from tqdm import tqdm

# # 源文件路径
# # PWD: e:\自学代码\搜推算法\deepmatch\DeepMatch\examples
# path = './Tenrec/QB-video.csv'

# # 分割后的子文件存储路径

# workspace = './Tenrec/ctr_task'

# with open(path, 'r', newline='') as file:
#     csvreader = csv.reader(file)
#     a = next(csvreader)
#     #print(a)
#     i = j = 0
#     for row in csvreader:
#         # 每 10000 个就 j 加 1，然后就有一个新的文件名。
#         if i % 1000000 == 0:
#             j += 1
#             print("完成第{}个100w数据".format(j-1))
# #
#         csv_path = os.path.join(workspace, 'QK_video1M.csv')
#         user_id = int(row[0])
#         if user_id < 1000017:
# #         # 不存在此文件的时候，就创建
#             if not os.path.exists(os.path.dirname(csv_path)):
#                 os.makedirs(os.path.dirname(csv_path))
#                 with open(csv_path, 'w', newline='') as file:
#                     csvwriter = csv.writer(file)
#                     csvwriter.writerow(row)
#                 i += 1
#     #         # 存在的时候就往里面添加
#             else:
#                 with open(csv_path, 'a', newline='') as file:
#                     csvwriter = csv.writer(file)
#                     csvwriter.writerow(row)
#                 i += 1

# path = os.path.join(workspace, 'QK_video1M.csv')
# source_data = pd.read_csv(path)
# source_data.columns = ['user_id', 'item_id', 'click', 'follow', 'like', 'share', 'short_v', 'play_times', 'gender', 'age']
# pos_data = source_data[source_data.click.isin([1])]
# user_history = pos_data.groupby('user_id').item_id.apply(list).to_dict()
# del pos_data
# user_hist = {}
# user_target = {}
# for key, value in user_history.items():
#     if len(value) <= 10:
#         if len(value) > 1:
#             user_hist[key] = value[:-1] + (10 - len(value[:-1])) * [0]
#             user_target[key] = value[-1:]
#         else:
#             user_hist[key] = [0] * 10
#             user_target[key] = value
#     else:
#         user_hist[key] = value[:10]
#         user_target[key] = value[10:]
# del_list = []
# for key, value in tqdm(user_hist.items()):
#     for v in value:
#         del_list.append([key, v])

# def del_data(s_data, user_hist, i):
#     print('++++++++{}+++++++'.format(i))
#     new = []
#     for user, value in tqdm(user_hist.items()):
#         if user > i * 10000 and user <= (i+1) * 10000:
#             data = s_data[s_data['user_id'] == user]
#             tmp_data = data[~data.item_id.isin(value)].values.tolist()
#             new.extend(tmp_data)
#     return new

# max_len = 1100000
# times = max_len // 10000
# new_list = []

# for i in range(times):
#     print("+++++++++times{}+++++++++".format(i))
#     data = source_data[(source_data['user_id'] > i * 10000) & (source_data['user_id'] <= (i+1) * 10000)]
#     new = del_data(data, user_hist, i)
#     new_list.extend(new)
#     # if len(new_list) == 100000 or i == times - 1:
# new_data = pd.DataFrame(new_list, columns=source_data.columns)

# hist_1, hist_2, hist_3, hist_4, hist_5, hist_6, hist_7, hist_8, hist_9, hist_10 = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
# for user, value in tqdm(user_hist.items()):
#     hist_1[user] = value[0]
#     hist_2[user] = value[1]
#     hist_3[user] = value[2]
#     hist_4[user] = value[3]
#     hist_5[user] = value[4]
#     hist_6[user] = value[5]
#     hist_7[user] = value[6]
#     hist_8[user] = value[7]
#     hist_9[user] = value[8]
#     hist_10[user] = value[9]

# for i in range(1, 11):
#     new_data['hist_{}'.format(i)] = new_data['user_id']
#     # new_data['hist_{}'.format(i)] = new_data['hist_{}'.format(i)].map(hist)
# new_data['hist_1'] = new_data['hist_1'].map(hist_1)
# new_data['hist_2'] = new_data['hist_2'].map(hist_2)
# new_data['hist_3'] = new_data['hist_3'].map(hist_3)
# new_data['hist_4'] = new_data['hist_4'].map(hist_4)
# new_data['hist_5'] = new_data['hist_5'].map(hist_5)
# new_data['hist_6'] = new_data['hist_6'].map(hist_6)
# new_data['hist_7'] = new_data['hist_7'].map(hist_7)
# new_data['hist_8'] = new_data['hist_8'].map(hist_8)
# new_data['hist_9'] = new_data['hist_9'].map(hist_9)
# new_data['hist_10'] = new_data['hist_10'].map(hist_10)
# new_data.to_csv('./ctr_data_1M.csv', header=True, index=False)



# 读取数据

In [ ]:
try:
    data = pd.read_csv('./ctr_data_1M.csv')
    
    data.rename(columns={'item_id': 'movie_id'}, inplace=True) #套用框架用movie_id偷懒一下

    # 补充缺失字段 (原 ML-1M 代码依赖 timestamp, genres 等)
    # timestamp: Tenrec 数据无时间戳，但一般负采样生成的数据在同一用户内相对有序。
    # 这里简单给一个递增序列作为伪时间戳，preprocess.py 依赖它排序
    #更新：观察数据集特点和preprocess作用后发现这部分没用hhhhh
    data['timestamp'] = range(len(data))
    # genres: Tenrec 只提供了 item_id，没有提供物品类别。
    # 为了简化改动，暂时给一个默认值，后续在 config 里把 genres 删掉。
    data['genres'] = 'General' 
    # occupation, zip: Tenrec 用户画像只有 gender, age。
    data['occupation'] = 0
    data['zip'] = 0

    #！！！！！
    data['rating'] = data['click'] # 将点击行为视为评分/正反馈

    # 原 ML-1M 数据全是“交互过”的数据。Tenrec 数据包含 click=0/1。
    # DSSM的负样本采样策略是 in batch, 不需要保留没点击的
    data = data[data['click'] == 1].copy()

    print("Tenrec 数据加载成功")
    print(data.head())

except FileNotFoundError:
    print("错误：未找到 ctr_data_1M.csv 文件。请确保已运行 data_process 脚本生成数据。")

Tenrec 数据加载成功
    user_id  movie_id  click  follow  like  share short_v  play_times  gender  \
19     2138   1446867      1       0     0      0       1           3       0   
20     2138   1831290      1       0     0      0       0           4       0   
21     2138   1744473      1       0     0      0       0           2       0   
22     2138   1597321      1       0     0      0       0           2       0   
27     2138   1356663      1       0     0      0       0           2       0   

    age  ...   hist_6   hist_7   hist_8   hist_9  hist_10  timestamp   genres  \
19    0  ...  1524428  1514403  1428355  1452826  3300984         19  General   
20    0  ...  1524428  1514403  1428355  1452826  3300984         20  General   
21    0  ...  1524428  1514403  1428355  1452826  3300984         21  General   
22    0  ...  1524428  1514403  1428355  1452826  3300984         22  General   
27    0  ...  1524428  1514403  1428355  1452826  3300984         27  General   

    occupati

# 构建特征列，训练模型，导出embedding

In [ ]:
# === 修改特征列表以充分利用 Tenrec 数据集 ===
# User侧特征: user_id, gender, age
# Item侧特征: movie_id (item_id), short_v (视频类型 短视频/非短视频), play_times (播放次数（理解成热度？）
sparse_features = ["user_id", "movie_id", "gender", "age", "short_v", "play_times"]
SEQ_LEN = 10 # Tenrec 数据集预处理生成了10个历史行为（因为hist1到hist10）
negsample = 0

In [ ]:
# 1. Label Encoding & Profile Extraction

feature_max_idx = {}
for feature in sparse_features:
    lbe = LabelEncoder()
    # 这一句在内存中发生了两件事：
    # 1. lbe.fit: 扫描 data['movie_id']，记住所有的原始ID（比如 'MovieA', 'MovieB'），存入 lbe.classes_
    # 2. transform + 1: 把 data['movie_id'] 这一列的原始字符串替换成了数字 [1, 2, ...]
    # String/Float -> Int, 并且 +1 (0留给padding)
    data[feature] = lbe.fit_transform(data[feature]) + 1
    # 记录最大值，用于定义 Embedding 矩阵的大小
    feature_max_idx[feature] = data[feature].max() + 1
    
    #用户历史行为序列（定长10，和ml-1m不一样）
    # 特殊处理：hist_1 到 hist_10 也是 movie_id，必须用同一个 LabelEncoder 转换
    # 确保用户的“历史观看记录（hist_1...hist_10）”和当前的“目标物品（movie_id）”使用完全相同的数字编码。
    if feature == "movie_id":
        for i in range(1, 11):
            col_name = f'hist_{i}'
            # 创建映射字典: 原始ID -> 编码ID
            # lbe.classes_ 依然是 ['MovieA', 'MovieB']
            # lbe.transform(lbe.classes_) 得到 [0, 1]
            # + 1 得到 [1, 2]
            # zip 组合起来：{'MovieA': 1, 'MovieB': 2}
            mapping = dict(zip(lbe.classes_, lbe.transform(lbe.classes_) + 1)) #建立一个查字典的规则表。结构大概是：{'泰坦尼克号': 1, '阿凡达': 2, ...}。
            # map转换，填充0，转int
            # data['hist_i'] 里面现在还是原始ID（如 'MovieA'）
            # 用上面的 mapping 去查：'MovieA' -> 1
            '''
            虽然逻辑是对的，但要所有在 hist_1...hist_10 里出现的电影，都必须在 movie_id 这一列里出现过。
            如果有一个冷门电影，用户看过（出现在 History 里），但在当前的 movie_id 列（也就是样本的 Target 列）里从未出现过，那么：
            这个冷门电影就会被映射成 0，导致模型在训练时无法看到这个电影的任何信息。

            会过分打压冷门物品的表现，后续可以考虑改进这一点。
            '''

            '''
            "如果一个物品在整个训练集中作为 Target（被点击的目标）一次都没出现过，
            那么模型就没有机会去学习它的 Embedding（因为没有梯度回传）。
            既然它的向量是随机初始化的‘废向量’，不如直接填 0，避免干扰。
            '''


# 提取用户画像 (User Profile)
# 包含: user_id, gender, age
user_profile = data[["user_id", "gender", "age"]].drop_duplicates('user_id')

# 提取物品画像 (Item Profile)
# 包含: movie_id, short_v, play_times
# 注意：这里加入了 short_v 和 play_times，后续做召回检索时会用到
item_profile = data[["movie_id", "short_v", "play_times"]].drop_duplicates('movie_id')

print('特征编码完成')
print('User Profile:', user_profile.shape)
print('Item Profile:', item_profile.shape)
print('=======================================')

特征编码完成
User Profile: (650, 3)
Item Profile: (21994, 3)


In [40]:
print("1. 数据集形状 (Rows, Columns):", data.shape)
print("\n2. 数据基本信息 (Type, Memory):")
print(data.info())

1. 数据集形状 (Rows, Columns): (57063, 25)

2. 数据基本信息 (Type, Memory):
<class 'pandas.core.frame.DataFrame'>
Int64Index: 57063 entries, 19 to 142114
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     57063 non-null  int64 
 1   movie_id    57063 non-null  int64 
 2   click       57063 non-null  int64 
 3   follow      57063 non-null  int64 
 4   like        57063 non-null  int64 
 5   share       57063 non-null  int64 
 6   short_v     57063 non-null  int32 
 7   play_times  57063 non-null  int64 
 8   gender      57063 non-null  int64 
 9   age         57063 non-null  int64 
 10  hist_1      57063 non-null  int32 
 11  hist_2      57063 non-null  int32 
 12  hist_3      57063 non-null  int32 
 13  hist_4      57063 non-null  int32 
 14  hist_5      57063 non-null  int32 
 15  hist_6      57063 non-null  int32 
 16  hist_7      57063 non-null  int32 
 17  hist_8      57063 non-null  int32 
 18  hist_9      57063 n

In [ ]:
print("\n3. 数值统计分布 (Mean, Min, Max):")
print(data.describe().round(2))


3. 数值统计分布 (Mean, Min, Max):
        user_id  movie_id    click    follow      like     share   short_v  \
count  57063.00  57063.00  57063.0  57063.00  57063.00  57063.00  57063.00   
mean     368.64  10348.22      1.0      0.00      0.02      0.00      1.54   
std      200.68   6039.35      0.0      0.03      0.15      0.04      0.50   
min        1.00      1.00      1.0      0.00      0.00      0.00      1.00   
25%      200.00   4838.50      1.0      0.00      0.00      0.00      1.00   
50%      454.00  10374.00      1.0      0.00      0.00      0.00      2.00   
75%      529.00  14900.00      1.0      0.00      0.00      0.00      2.00   
max      650.00  21994.00      1.0      1.00      1.00      1.00      3.00   

       play_times    gender       age  ...    hist_5    hist_6    hist_7  \
count    57063.00  57063.00  57063.00  ...  57063.00  57063.00  57063.00   
mean         2.20      2.02      3.49  ...   5998.29   7238.87   6248.92   
std          1.96      0.43      1.06  .

In [ ]:
print("\n4. 关键特征的唯一值数量 (Unique Counts):")
# 可以看一下Embedding 矩阵的大小
feature_cols111 = ['user_id', 'movie_id', 'gender', 'age', 'short_v', 'play_times']
for col in feature_cols111:
    if col in data.columns:
        print(f"{col}: {data[col].nunique()} unique values")


4. 关键特征的唯一值数量 (Unique Counts):
user_id: 650 unique values
movie_id: 21994 unique values
gender: 3 unique values
age: 8 unique values
short_v: 3 unique values
play_times: 68 unique values


In [43]:
print("\n5. 历史序列的填充情况:")
# 检查 hist_1 到 hist_10 中有多少比例是 0 (Padding)
hist_cols = [f'hist_{i}' for i in range(1, 11)]
zeros = (data[hist_cols] == 0).sum().sum()
total = data[hist_cols].size
print(f"Padding (0) 占比: {zeros/total:.2%}")


5. 历史序列的填充情况:
Padding (0) 占比: 23.45%


In [5]:
# 设置索引，方便查询
user_profile.set_index("user_id", inplace=True)
# item_profile.set_index("movie_id", inplace=True) # item_profile 暂时不设索引，保留 movie_id 列方便后续操作

print('用户画像样例:')
print(user_profile.head())
print('物品画像样例:')
print(item_profile.head())
print('=======================================')

用户画像样例:
         gender  age
user_id             
1             1    1
2             2    4
3             2    5
4             1    1
5             2    4
物品画像样例:
    movie_id  short_v  play_times
19      9513        2           3
20     12834        1           4
21     12533        1           2
22     11728        1           2
27      3933        1           2


In [ ]:
# 数据集划分
# 由于 ctr_data_1M.csv 包含Label和固定长度历史序列
# 所以不需要使用 preprocess.py 中的 gen_data_set 用滑动窗口生成历史序列
# 直接使用 sklearn 进行随机切分即可。

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.2, random_state=2022)

print('数据集切分 (Dataset Splitting)')
print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
print('=======================================')

数据集切分 (Dataset Splitting)
Train size: 45650
Test size: 11413


In [ ]:
# 生成模型输入 (Generate Model Input)
# 跳过preprocess.py的gen_model_input，学习它的逻辑把 DataFrame 转换成 DeepMatch/Keras 能够接受的字典形式
import numpy as np

def get_model_input(df):
    # 1. 提取基础 Sparse 特征
    model_input = {
        "user_id": df['user_id'].values,
        "movie_id": df['movie_id'].values,
        "gender": df['gender'].values,
        "age": df['age'].values,
        "short_v": df['short_v'].values,  
        "play_times": df['play_times'].values, 
    }
    
    # 2. 提取历史行为序列 (hist_1 ... hist_10)
    # 将这 10 列合并成一个 (N, 10) 的矩阵
    hist_cols = [f'hist_{i}' for i in range(1, 11)]
    history_matrix = df[hist_cols].values
    # User History 也是 Movie ID
    model_input['hist_movie_id'] = history_matrix
    
    # 3. 计算实际序列长度 (hist_len)
    # 统计每一行中不为 0 的元素个数，作为变长序列的真实长度
    model_input['hist_len'] = np.count_nonzero(history_matrix, axis=1)
    
    # 4. 提取标签 (Click)
    label = df['click'].values
    
    return model_input, label

train_model_input, train_label = get_model_input(train_data)
test_model_input, test_label = get_model_input(test_data)

print('生成模型输入完成')
print("Train Input - user_id sample:", train_model_input['user_id'][:2])
print("Train Input - short_v sample:", train_model_input['short_v'][:2])
print("Train Input - hist sample:", train_model_input['hist_movie_id'][:1])
print('=======================================')

生成模型输入完成
Train Input - user_id sample: [500  17]
Train Input - short_v sample: [1 2]
Train Input - hist sample: [[19958     0     0 19932     0  4574 13326  6288  2272  8473]]


In [ ]:
# 2.Feature Config 配置

embedding_dim = 32

# 构建用户塔特征 (User Tower Features)
user_feature_columns = [
    # 1). 稀疏特征 (Sparse Categorical Features)
    SparseFeat('user_id', feature_max_idx['user_id'], 16),
    SparseFeat("gender", feature_max_idx['gender'], 16),
    SparseFeat("age", feature_max_idx['age'], 16),
    
    # 2). 变长序列特征 (Sequence Features)
    # 注意 maxlen=10，对应 hist_1 到 hist_10
    # embedding_name="movie_id" 意味着历史行为里的 ID 和 items 里的 movie_id 共享同一张 Embedding 表
    VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                embedding_name="movie_id"), 
                                maxlen=10, combiner='mean', length_name='hist_len'),
]

# 构建物品塔特征 (Item Tower Features)
# 加入了 short_v 和 play_times
item_feature_columns = [
    SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
    SparseFeat('short_v', feature_max_idx['short_v'], embedding_dim),
    SparseFeat('play_times', feature_max_idx['play_times'], embedding_dim),
]

print("特征配置完成")

特征配置完成


In [9]:
user_feature_columns

[SparseFeat(name='user_id', vocabulary_size=651, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BFA00>, embedding_name='user_id', group_name='default_group', trainable=True),
 SparseFeat(name='gender', vocabulary_size=4, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BF310>, embedding_name='gender', group_name='default_group', trainable=True),
 SparseFeat(name='age', vocabulary_size=9, embedding_dim=16, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BFDF0>, embedding_name='age', group_name='default_group', trainable=True),
 VarLenSparseFeat(sparsefeat=SparseFeat(name='hist_movie_id', vocabulary_size=

In [10]:
item_feature_columns

[SparseFeat(name='movie_id', vocabulary_size=21995, embedding_dim=32, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BFC70>, embedding_name='movie_id', group_name='default_group', trainable=True),
 SparseFeat(name='short_v', vocabulary_size=4, embedding_dim=32, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BF640>, embedding_name='short_v', group_name='default_group', trainable=True),
 SparseFeat(name='play_times', vocabulary_size=69, embedding_dim=32, use_hash=False, vocabulary_path=None, dtype='int32', embeddings_initializer=<tensorflow.python.keras.initializers.initializers_v1.RandomNormal object at 0x000001D4A26BF6A0>, embedding_name='play_times', group_name='default_group', trainable=True)]

In [ ]:
# 4). 负采样配置 (Negative Sampler Config)
from collections import Counter

# 统计训练集中每个电影出现的次数
# 用于 生成采样概率分布，用于 sampled softmax
train_counter = Counter(train_model_input['movie_id'])

# 找到 movie_id 对应的 Feature Column (防止 后期改动 feature 导致顺序变化导致取错)
movie_id_col = [x for x in item_feature_columns if x.name == 'movie_id'][0]

# 生成一个列表，索引是 movie_id，值是该 movie_id 出现的次数
# vocabulary_size 是 feature_max_idx['movie_id'] (即最大ID+1)
item_count = [train_counter.get(i,0) for i in range(movie_id_col.vocabulary_size)]

print(f"Item Count Length: {len(item_count)}")
print("前5个物品的频次:", item_count[:5])

Item Count Length: 21995
前5个物品的频次: [0, 1, 0, 1, 1]


In [12]:
train_counter

Counter({14125: 64,
         14121: 56,
         14331: 51,
         14324: 49,
         12419: 48,
         11717: 48,
         14647: 47,
         18767: 46,
         14815: 46,
         9183: 45,
         14286: 45,
         14872: 45,
         5891: 43,
         8551: 43,
         14636: 42,
         14381: 42,
         6658: 41,
         4727: 40,
         12930: 40,
         12115: 40,
         3687: 40,
         14758: 38,
         14109: 37,
         14756: 37,
         14160: 37,
         3887: 36,
         14320: 36,
         14233: 36,
         6328: 35,
         14103: 35,
         14314: 35,
         5534: 35,
         14598: 34,
         6271: 34,
         3683: 34,
         6977: 34,
         5560: 34,
         5225: 34,
         9351: 34,
         14900: 33,
         4623: 33,
         14123: 33,
         4408: 33,
         15562: 32,
         14344: 32,
         4443: 32,
         4640: 32,
         8142: 32,
         14638: 32,
         5056: 32,
         7435: 31,
  

In [ ]:
item_count
#可能有些电影在测试集中所以在这里count是0

[0,
 1,
 0,
 1,
 1,
 1,
 3,
 1,
 2,
 0,
 1,
 1,
 1,
 1,
 7,
 1,
 1,
 10,
 1,
 1,
 3,
 3,
 25,
 2,
 1,
 1,
 1,
 0,
 8,
 2,
 0,
 1,
 1,
 3,
 2,
 1,
 2,
 1,
 1,
 3,
 2,
 4,
 1,
 2,
 3,
 1,
 1,
 3,
 2,
 10,
 1,
 1,
 1,
 14,
 7,
 1,
 0,
 2,
 3,
 0,
 1,
 1,
 1,
 1,
 1,
 15,
 2,
 1,
 3,
 4,
 1,
 1,
 1,
 1,
 0,
 12,
 1,
 1,
 1,
 0,
 7,
 1,
 0,
 1,
 2,
 4,
 1,
 1,
 1,
 2,
 1,
 2,
 6,
 0,
 1,
 0,
 1,
 4,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 2,
 1,
 1,
 3,
 0,
 1,
 1,
 3,
 1,
 2,
 1,
 4,
 0,
 1,
 3,
 1,
 5,
 3,
 1,
 2,
 10,
 1,
 1,
 0,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 5,
 3,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 15,
 2,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 12,
 5,
 0,
 17,
 13,
 21,
 1,
 3,
 1,
 6,
 0,
 15,
 1,
 0,
 1,
 8,
 0,
 8,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 3,
 1,
 0,
 2,
 5,
 8,
 1,
 0,
 1,
 3,
 1,
 1,
 4,
 2,
 2,
 1,
 0,
 6,
 2,
 1,
 1,
 3,
 1,
 1,
 1,
 1,
 1,
 3,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 4,
 1,
 1,
 3,
 1,
 1,
 4,
 1,
 4,
 5,
 1,
 3,
 1,
 1,
 1,
 2,
 4,
 1,
 0,

In [14]:
## 定义采样器
sampler_config = NegativeSampler('inbatch',num_sampled=255,item_name="movie_id",item_count=item_count)

In [15]:
# 3.Define Model and train
#定义模型与训练 (Define Model and Train)
#这一阶段的目标是让模型学习如何将 User 和 Item 映射到同一个向量空间，使得用户和他感兴趣的物品距离更近。

import tensorflow as tf
#禁用了 TensorFlow 2.x 默认的 Eager Execution（动态图模式），强制使用 Graph Execution（静态图模式）。
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
'''
原因：
In-Batch Softmax 的特殊性：
    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。
    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，
    这对于大规模分类和负采样训练至关重要。
库的兼容性：
    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，
    关闭 Eager 模式能保证最大程度的兼容和稳定。
'''

'\n原因：\nIn-Batch Softmax 的特殊性：\n    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。\n    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，\n    这对于大规模分类和负采样训练至关重要。\n库的兼容性：\n    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，\n    关闭 Eager 模式能保证最大程度的兼容和稳定。\n'

##### 至此完成了准备数据（preprocess）、配置特征列（SparseFeat/VarLenSparseFeat）、配置负采样策略（NegativeSampler

#### 开始以构建并训练DSSM

In [ ]:
#初始化 DSSM 模型
# Tenrec 数据量较大，User History 是固定长度10，输入维度和模型参数需要适配
#embedding_dim = 32
model = DSSM(user_feature_columns, item_feature_columns,
             user_dnn_hidden_units=(128, 64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim),
             loss_type='softmax',
             sampler_config=sampler_config)

# 编译模型
model.compile(optimizer="adam", loss=sampledsoftmaxloss)

# 开始训练
# 注意：train_label 在这里其实不参与 Loss 计算 (In-Batch Softmax 自带 Loss)，但 Keras fit 需要它
history = model.fit(train_model_input, train_label,
                    batch_size=256, epochs=20, verbose=1, validation_split=0.0, )


Train on 45650 samples
Epoch 1/20
45650/45650 [==============================] - 2s 49us/sample - loss: 5.7227
Epoch 2/20
45650/45650 [==============================] - 2s 41us/sample - loss: 5.1932
Epoch 3/20
45650/45650 [==============================] - 2s 42us/sample - loss: 4.4546
Epoch 4/20
45650/45650 [==============================] - 2s 42us/sample - loss: 4.4546
Epoch 4/20
45650/45650 [==============================] - 2s 43us/sample - loss: 3.9642
Epoch 5/20
45650/45650 [==============================] - 2s 43us/sample - loss: 3.9642
Epoch 5/20
45650/45650 [==============================] - 2s 43us/sample - loss: 3.7082
Epoch 6/20
45650/45650 [==============================] - 2s 43us/sample - loss: 3.7082
Epoch 6/20
45650/45650 [==============================] - 2s 44us/sample - loss: 3.5441
Epoch 7/20
45650/45650 [==============================] - 2s 44us/sample - loss: 3.5441
Epoch 7/20
45650/45650 [==============================] - 2s 43us/sample - loss: 3.4244
Epoch 8/2

In [ ]:
# 4. Generate user features for testing and full item features for retrieval
# 提取向量 (Generate Embeddings)

# 用户侧使用测试集的用户数据
test_user_model_input = test_model_input

# 物品侧使用全量数据库
all_item_model_input = {
    "movie_id": item_profile['movie_id'].values,
    "short_v": item_profile['short_v'].values,
    "play_times": item_profile['play_times'].values
}

print("预测数据准备完成")
print("All Item Input Keys:", all_item_model_input.keys())

预测数据准备完成
All Item Input Keys: dict_keys(['movie_id', 'short_v', 'play_times'])


In [ ]:
#model 是一个完整的计算图。
#我们可以定义一个新的 user_embedding_model，它的输入和原模型一样，但输出截断在 User 塔的最后一层 (model.user_embedding)。Item 侧同理。
user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
##拥有了两个独立的模型：
#输入用户信息 -> 输出 32维 User 向量。
#输入物品信息 -> 输出 32维 Item 向量。

In [31]:
#批量推断 (Batch Inference)
#这步操作叫做 “离线向量化”（Offline Vectorization），是召回模型从“训练阶段”走向“服务/预测阶段”的分水岭
#核心目的：把训练好的神经网络，变成两个可以查表的 “向量矩阵”。
#训练时 (Training): 我们是 Pair-wise 的（成对输入）。模型就像一个严厉的教官，同时看着 User 和 Item，计算它们匹不匹配，然后调整它们各自的权重。 模型结构：User Input + Item Input -> Score -> Loss
#这步推断 (Inference): 我们现在要把教官这一对“拆散”。
    #User Side: 让所有用户排好队，一个个通过 User 塔，算出他们每个人的“兴趣向量” (user_embs)。
    #Item Side: 让全库所有电影排好队，一个个通过 Item 塔，算出它们每个电影的“特征向量” (item_embs)。

# 1. 计算用户向量
# Input: 测试集里的用户特征 (test_user_model_input)
# Action: 跑一遍 User Embedding Model (DSSM 的左塔)
# Output: user_embs (Shape: [用户数, 32])
# 含义: 每一行代表一个用户，即使这个用户很复杂（有画像、有几百条历史行为），现在都被压缩成了这 32 个数字。
user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)


# 2. 计算物品向量
# Input: 全库所有电影的特征 (all_item_model_input)
# Action: 跑一遍 Item Embedding Model (DSSM 的右塔)
# Output: item_embs (Shape: [电影总数, 32])
# 含义: 每一行代表一部电影，这 32 个数字浓缩了电影的风格、题材等信息。
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

# 注意：Faiss 建索引时需要 float32 类型，predict 返回的通常已经是 float32，但为了保险可以确认一下
# 如果遇到 Faiss 报错类型不匹配，可以手动 .astype('float32')

print(f"User Embeddings Shape: {user_embs.shape}")
print(f"Item Embeddings Shape: {item_embs.shape}")

User Embeddings Shape: (11413, 32)
Item Embeddings Shape: (21994, 32)


# 使用faiss进行ANN查找并评估结果

这几步是在进行全流程的实战演练：模拟一个真实的线上召回系统是如何工作的，并给它打分。

In [20]:
import numpy as np
import faiss
from tqdm import tqdm
from deepmatch.utils import recall_N

In [32]:
#1. 建库 (Indexing) —— "把图书馆建好"
#第一步：构建向量索引 (Build Index with Faiss)
#这一步相当于把所有商品都存入了向量数据库中，准备等待用户的查询。

# 1. 创建索引
# IndexFlatIP: "Flat"表示暴力搜索（不压缩，保证精度），"IP"代表 Inner Product（内积）。
# 在深度学习中，内积通常用于衡量向量相似度（类似于余弦相似度，如果向量归一化过的话）。
index = faiss.IndexFlatIP(embedding_dim)

## 2. 将物品向量加入索引
# item_embs: 之前算好的所有电影的 Embedding 矩阵，形状是 (电影总数, 32)。
# faiss.normalize_L2(item_embs)
index.add(item_embs)

In [33]:
#2. 检索 (Retrieval) —— "读者来借书"
#第二步：进行批量检索 (Batch Search)
#这里直接用所有测试集用户的向量去数据库里“搜”他们可能喜欢的电影。

# faiss.normalize_L2(user_embs)
## D: Distance (这里是内积得分)，形状 (用户数, 100)
# I: Index (物品的ID索引)，形状 (用户数, 100)
# search(user_embs, 100): 对每个用户向量，找出内积最大的 Top 100 个物品。
D, I = index.search(np.ascontiguousarray(user_embs), 100) # 将搜索结果从50改为100，以便后续计算Recall@100
#np.ascontiguousarray：Faiss 要求输入数据在内存中是连续的 C-order 数组，这是一个常见的工程细节。
#结果：对于第 i 个用户，I[i] 保存了推荐给他的 100 个电影在 item_profile 中的行号（索引）。

#注意“计算内积这件事，被外包给了 Faiss”
#index.search 内部在做什么？ 它在做疯狂的矩阵乘法（Matrix Multiplication）。
#数学上Scores=UserMatrix×ItemMatrix^T
#user_embs 是 U×K（用户数 × 32）
#item_embs 是 I×K（物品数 × 32）
#它们相乘得到 U×I 的分数矩阵。
#Faiss 的厉害之处： 如果让你写 for 循环去算，或者用普通的 numpy 算，面对百万级商品可能会很慢。 Faiss 用了极度优化的 C++ 代码、SIMD 指令集甚至 GPU 加速，在几毫秒内就能算完这些内积，并且顺便帮你排个序，只把最大的 100 个分数值（D）和对应的物品编号（I）吐给你。

In [34]:
D

array([[0.71931314, 0.7103748 , 0.6959859 , ..., 0.60924023, 0.6084787 ,
        0.60841393],
       [0.7001335 , 0.69536275, 0.68308467, ..., 0.5788652 , 0.5787866 ,
        0.5786697 ],
       [0.7696444 , 0.7513067 , 0.741929  , ..., 0.65434504, 0.6526865 ,
        0.65218806],
       ...,
       [0.6662551 , 0.6585419 , 0.6578724 , ..., 0.59937024, 0.59916955,
        0.5986651 ],
       [0.6765922 , 0.6757483 , 0.6636558 , ..., 0.6059669 , 0.6054879 ,
        0.6052366 ],
       [0.66514724, 0.62472695, 0.61284435, ..., 0.5083329 , 0.5082934 ,
        0.5082763 ]], dtype=float32)

In [35]:
I

array([[10332, 10329, 10319, ...,   513,  1199,  1017],
       [10847,  6792, 10848, ...,  6456,  7040,  1841],
       [19168, 19159, 19169, ..., 19056,   769, 19167],
       ...,
       [18712,  2162, 18669, ..., 18733, 18686, 18740],
       [ 5355,  5446,  5390, ...,  5383,  5284,  5227],
       [ 8534,  8532,  8522, ...,   117,   889,   467]], dtype=int64)

In [ ]:
#3. 评估 (Evaluation) —— "看看推得准不准"
#第三步：评估效果 (Evaluation)

# 1. 准备真实标签字典 (Ground Truth)
# 在我们新的 train_test_split 划分逻辑中，test_data 就是我们的测试集 DataFrame
# 每一行包含：user_id, movie_id (真实点击的那个)

# 我们构建一个字典：{user_id: {item_id1, item_id2, ...}}
# 虽然这里做了去重，并且看起来每个用户只有一行测试数据，但为了代码健壮性（万一测试集同一个用户有多条记录），用 set 存储
test_true_label = {}
for user, item in zip(test_data['user_id'], test_data['movie_id']):
    if user not in test_true_label:
        test_true_label[user] = set()
    test_true_label[user].add(item)

#输出：test_true_label 变成了一个以用户ID为 Key，以“该用户在测试集里看过的所有电影集合”为 Value 的字典。
#例如：
#{
#     'UserA': {'Movie1', 'Movie2'},
#     'UserB': {'Movie3'}
# }
#所以这几行代码是为了后续能飞快地计算命中率而构建的高效查询索引。

print(f"测试集用户数: {len(test_true_label)}")


测试集用户数: 511


In [37]:
# 2. 遍历每个用户进行对比
# 这里一定要注意：上一步 index.search(..., 100) 必须把 k 改为 100

# 定义存储结果的容器
metrics = {
    50: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []},
    100: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []}
}

# 预先构建 Answer Set，提升速度
# key是用户ID，value是该用户在测试集中实际看过的电影ID的 Set (O(1) lookup)
test_true_label = {}
for user, item in zip(test_data['user_id'], test_data['movie_id']):
    if user not in test_true_label:
        test_true_label[user] = set()
    test_true_label[user].add(item)


# 这里的 item_profile['movie_id'] 需要是一个numpy array，以便通过索引访问
all_item_ids = item_profile['movie_id'].values

for i, uid in tqdm(enumerate(test_user_model_input['user_id'])):
    try:
        # I[i]得到的是 TopK 的 Index (0, 1, 2...)
        # 我们需要把它还原成真实的 movie_id
        pred_full = all_item_ids[I[i]]
        
        # 获取用户真实看过的电影列表 
        if uid not in test_true_label:
            continue
            
        true_items = test_true_label[uid] 

        # 分别计算 @50 和 @100 的指标
        for K in [50, 100]:
            # 截取前 K 个推荐结果
            pred_k = pred_full[:K]
            
            # --- 1. 基础命中统计 (Intersection) ---
            # 预测对了几个？
            hits = set(pred_k) & true_items
            hit_count = len(hits)

            # --- 2. Hit Rate ---
            if hit_count > 0:
                metrics[K]['hit'] += 1
            
            # --- 3. Recall (召回率) ---
            recall_val = hit_count / len(true_items)
            metrics[K]['recall'].append(recall_val)
            
            # --- 4. Precision (精确率) ---
            precision_val = hit_count / K
            metrics[K]['precision'].append(precision_val)
            
            # --- 5. F1-Score ---
            if (precision_val + recall_val) > 0:
                f1_val = 2 * (precision_val * recall_val) / (precision_val + recall_val)
            else:
                f1_val = 0
            metrics[K]['f1'].append(f1_val)

            # --- 6. NDCG (归一化折损累计增益) ---
            dcg_k = 0
            for rank, item in enumerate(pred_k):
                if item in true_items:
                    dcg_k += 1.0 / np.log2(rank + 2)
            
            idcg_k = 0
            for rank in range(min(len(true_items), K)):
                idcg_k += 1.0 / np.log2(rank + 2)
            
            if idcg_k > 0:
                metrics[K]['ndcg'].append(dcg_k / idcg_k)
            else:
                metrics[K]['ndcg'].append(0)

    except Exception as e:
        print(f"Error at index {i}: {e}") # 打印出错信息

# 打印最终报告
print("-" * 30)
for K in [50, 100]:
    print(f"Metrics @ {K}:")
    print(f"  Recall   : {np.mean(metrics[K]['recall']):.4f}")
    print(f"  Precision: {np.mean(metrics[K]['precision']):.4f}")
    print(f"  NDCG     : {np.mean(metrics[K]['ndcg']):.4f}")
    print(f"  F1-Score : {np.mean(metrics[K]['f1']):.4f}")
    print(f"  Hit Rate : {metrics[K]['hit'] / len(test_user_model_input['user_id']):.4f}")
    print("-" * 30)

0it [00:00, ?it/s]

11413it [00:01, 10022.21it/s]

------------------------------
Metrics @ 50:
  Recall   : 0.0197
  Precision: 0.0134
  NDCG     : 0.0175
  F1-Score : 0.0113
  Hit Rate : 0.3885
------------------------------
Metrics @ 100:
  Recall   : 0.0403
  Precision: 0.0144
  NDCG     : 0.0260
  F1-Score : 0.0161
  Hit Rate : 0.6195
------------------------------
